In [10]:
# imports
import pandas as pd
import ast


In [2]:
# read files
kor_patients = pd.read_csv('kor_merged_aggregated_1W_mean.csv')
neuro_patients = pd.read_csv('neuro_merged_aggregated_1W_mean.csv')

print("KOR patients head: ", kor_patients.head())
print("\nNEURO patients head: ", neuro_patients.head())

KOR patients head:        custom_id  patient_id examination_date  patient_age  patient_sex  \
0   91-2021-W26          91       2021-07-04        56.07          1.0   
1  114-2021-W23         114       2021-06-13        78.42          0.0   
2  231-2020-W28         231       2020-07-12        82.59          1.0   
3  316-2021-W40         316       2021-10-10        79.53          1.0   
4  316-2021-W41         316       2021-10-17        79.54          1.0   

   diagnosis_id                                 descriptive_result  \
0           NaN  {'values': ['Hemoliza próbki badanej. Wynik po...   
1           NaN                                                NaN   
2           NaN                                                NaN   
3           NaN                                                NaN   
4           NaN  {'values': ['Silna hemoliza próbki badanej. Pr...   

                                    examination_type        MCH MCH-unit  ...  \
0  {'values': ['Morfologia', 'Bia

In [4]:
print("KOR patients:", kor_patients.columns.tolist())
print("\nNEURO patients:", neuro_patients.columns.tolist())

KOR patients: ['custom_id', 'patient_id', 'examination_date', 'patient_age', 'patient_sex', 'diagnosis_id', 'descriptive_result', 'examination_type', 'MCH', 'MCH-unit', 'MCH-norm', 'HGB', 'HGB-unit', 'HGB-norm', 'MCV', 'MCV-unit', 'MCV-norm', 'RBC', 'RBC-unit', 'RBC-norm', 'MCHC', 'MCHC-unit', 'MCHC-norm', 'PLT', 'PLT-unit', 'PLT-norm', 'HCT', 'HCT-unit', 'HCT-norm', 'WBC', 'WBC-unit', 'WBC-norm', 'RDW', 'RDW-unit', 'RDW-norm', '%BAZO', '%BAZO-unit', '%BAZO-norm', 'BAZO', 'BAZO-unit', 'BAZO-norm', 'EO', 'EO-unit', 'EO-norm', '%EO', '%EO-unit', '%EO-norm', 'LYMPH', 'LYMPH-unit', 'LYMPH-norm', '%LYMPH', '%LYMPH-unit', '%LYMPH-norm', '%MONO', '%MONO-unit', '%MONO-norm', 'MONO', 'MONO-unit', 'MONO-norm', 'MPV', 'MPV-unit', 'MPV-norm', 'KREA', 'KREA-unit', 'KREA-norm', 'Na', 'Na-unit', 'Na-norm', 'K', 'K-unit', 'K-norm', '%IG', '%IG-unit', '%IG-norm', 'IG', 'IG-unit', 'IG-norm', 'eGFR-MDRD', 'eGFR-MDRD-unit', 'eGFR-MDRD-norm', 'eGFRCKD', 'eGFRCKD-unit', 'eGFRCKD-norm', 'NEUT', 'NEUT-unit', 

In [8]:
# check duplicates in neuro
id_col = 'custom_id'
neuro_duplicates = neuro_patients[neuro_patients.duplicated(subset=[id_col], keep=False)]
neuro_duplicates = neuro_duplicates.sort_values(by=[id_col])

print(f"Found {len(neuro_duplicates)} duplicates in NEURO.")

if len(neuro_duplicates) > 0:
    print("Duplicated rows:")
    display(neuro_duplicates)

neuro_dataset = neuro_patients.drop_duplicates(subset=[id_col], keep='first')

# check duplicates in kor
kor_duplicates = kor_patients[kor_patients.duplicated(subset=[id_col], keep=False)]
kor_duplicates = kor_duplicates.sort_values(by=[id_col])

print(f"Found {len(kor_duplicates)} duplicates in KOR.")

if len(kor_duplicates) > 0:
    print("Duplicated rows:")
    display(kor_duplicates)

kor_dataset = kor_patients.drop_duplicates(subset=[id_col], keep='first')


Found 2 duplicates in NEURO.
Duplicated rows:


,custom_id,patient_id,examination_date,patient_age,patient_sex,diagnosis_id,descriptive_result,examination_type,MCH,MCH-unit,...,ALT-norm,BUN,BUN-unit,BUN-norm,AST,AST-unit,AST-norm,P-LCR,P-LCR-unit,P-LCR-norm
754,194143-2017-W52,194143,2017-01-01,71.05,1,NaN,NaN,"{'values': ['Elektrolity', 'Morfologia', 'Krea...",30.0,pg,...,0.0,NaN,NaN,NaN,20.0,U/l,0.0,NaN,NaN,NaN
778,194143-2017-W52,194143,2017-12-31,72.05,1,NaN,NaN,"{'values': ['Morfologia', 'Kreatynina we krwi'...",30.0,pg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Found 0 duplicates in KOR.


In [13]:
# check for duplicates between files
merged_duplicates = pd.merge(kor_dataset, neuro_dataset, on=id_col, how='inner')
print(f"Found {len(merged_duplicates)} duplicates between KOR and NEURO datasets.")

# kolumny do porównania
cols_to_compare = [c for c in kor_dataset.columns if c != id_col]

merged = pd.merge(
    kor_dataset,
    neuro_dataset,
    on=id_col,
    how='inner',
    suffixes=('_kor', '_neuro')
)

# maska różnic (z obsługą NaN)
diff_mask = pd.DataFrame({
    col: ~(
        merged[f"{col}_kor"].fillna("NA") ==
        merged[f"{col}_neuro"].fillna("NA")
    )
    for col in cols_to_compare
})

# wiersze gdzie cokolwiek się różni
rows_with_diff = diff_mask.any(axis=1)

# wybieramy tylko kolumny gdzie są różnice (globalnie)
cols_with_any_diff = diff_mask.columns[diff_mask.any()].tolist()

# budujemy finalny widok
result = merged.loc[rows_with_diff, [id_col] +
    [f"{col}_kor" for col in cols_with_any_diff] +
    [f"{col}_neuro" for col in cols_with_any_diff]
]

display(result)

Found 559 duplicates between KOR and NEURO datasets.


,custom_id,patient_age_kor,descriptive_result_kor,examination_type_kor,MCH_kor,MCH-unit_kor,MCH-norm_kor,HGB_kor,HGB-unit_kor,HGB-norm_kor,...,ALT-norm_neuro,BUN_neuro,BUN-unit_neuro,BUN-norm_neuro,AST_neuro,AST-unit_neuro,AST-norm_neuro,P-LCR_neuro,P-LCR-unit_neuro,P-LCR-norm_neuro
0,10237-2021-W2,89.42,NaN,"{'values': ['Morfologia', 'Białko C-reaktywne'...",30.966667,pg,0.0,11.766667,g/dl,-1.0,...,0.0,17.5,mg/dl,0.0,27.0,U/l,0.0,36.733333,%,1.0
1,10237-2021-W3,89.44,NaN,"{'values': ['Morfologia', 'Kreatynina we krwi'...",30.600000,pg,0.0,10.466667,g/dl,-1.0,...,0.0,NaN,NaN,NaN,20.0,U/l,0.0,33.966667,%,0.0
2,10237-2021-W4,89.45,NaN,"{'values': ['Białko C-reaktywne', 'Elektrolity...",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.600000,%,0.0
3,12989-2020-W7,69.96,NaN,"{'values': ['Morfologia', 'Białko C-reaktywne'...",32.350000,pg,1.0,15.200000,g/dl,0.0,...,NaN,20.0,mg/dl,0.0,13.0,U/l,0.0,34.250000,%,1.0
4,12989-2020-W8,69.96,NaN,"{'values': ['Morfologia', 'Białko C-reaktywne'...",32.200000,pg,1.0,12.000000,g/dl,-1.0,...,0.0,15.0,mg/dl,0.0,14.0,U/l,0.0,36.200000,%,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,2241420-2021-W49,72.64,NaN,"{'values': ['Morfologia', 'Białko C-reaktywne'...",36.550000,pg,1.0,13.900000,g/dl,-1.0,...,1.0,35.5,mg/dl,1.0,108.0,U/l,1.0,39.650000,%,1.0
555,2242939-2021-W49,46.75,{'values': ['Hemoliza próbki badanej. Wynik po...,"{'values': ['Morfologia', 'Kreatynina we krwi'...",34.650000,pg,1.0,15.850000,g/dl,0.0,...,1.0,NaN,NaN,NaN,83.0,U/l,1.0,32.800000,%,1.0
556,2242939-2021-W50,46.76,NaN,"{'values': ['Morfologia', 'Kreatynina we krwi'...",34.760000,pg,1.0,15.020000,g/dl,0.0,...,NaN,10.0,mg/dl,0.0,NaN,NaN,NaN,33.260000,%,1.0
557,2244565-2021-W51,75.78,NaN,"{'values': ['Morfologia', 'Białko C-reaktywne'...",26.650000,pg,-1.0,7.400000,g/dl,-1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.700000,%,0.0


In [14]:
# dopasowanie jedostek w kolumnach unit

def clean_and_check_units(df, id_col='custom_id'):
    df = df.copy()
    unit_cols = [c for c in df.columns if c.endswith('-unit')]

    for col in unit_cols:
        mode = df[col].mode(dropna=True)

        if mode.empty:
            print(f"[WARN] {col}: brak wartości (same NaN)")
            continue

        most_common = mode[0]

        # znajdź "dziwne" wartości
        weird_mask = (df[col].notna()) & (df[col] != most_common)

        if weird_mask.any():
            print(f"\n[ALERT] {col}: znaleziono inne unity niż '{most_common}'")
            print(df.loc[weird_mask, [id_col, col]].head())

        # uzupełnij NaN
        df[col] = df[col].fillna(most_common)

    return df

kor_dataset = clean_and_check_units(kor_dataset)
neuro_dataset = clean_and_check_units(neuro_dataset)

[WARN] INR-unit: brak wartości (same NaN)
[WARN] WAPTT-unit: brak wartości (same NaN)
[WARN] INR-unit: brak wartości (same NaN)
[WARN] WAPTT-unit: brak wartości (same NaN)


In [19]:
# ekstrakcja descriptive_result

def extract_description(val):
    if pd.isna(val):
        return val

    try:
        parsed = ast.literal_eval(val)
        return " ".join(parsed.get("values", []))
    except:
        return val

# zastosowanie
kor_dataset['descriptive_result'] = kor_dataset['descriptive_result'].apply(extract_description)
neuro_dataset['descriptive_result'] = neuro_dataset['descriptive_result'].apply(extract_description)



NEURO descriptive_result:


0                                                  NaN
1                                                  NaN
2                                                  NaN
3                        Hemoliza. Hemoliza. Hemoliza.
4                                                  NaN
5                                                  NaN
6    Hemoliza próbki badanej. Proszę o ponowne pobr...
7                                                  NaN
8                                                  NaN
9                                                  NaN
Name: descriptive_result, dtype: str

In [40]:
def extract_values(df, col='examination_type'):
    values = set()

    for v in df[col].dropna():
        try:
            values.update(ast.literal_eval(v).get("values", []))
        except:
            pass

    return values

# wszystkie examination_type
all_exam_types = extract_values(kor_dataset) | extract_values(neuro_dataset)

print("Wszystkie examination_type values:")
print(sorted(all_exam_types))

values_cols = [
    'MCH', 'HGB', 'MCV', 'RBC', 'MCHC', 'PLT', 'HCT', 'WBC', 'RDW',
    '%BAZO', 'BAZO', 'EO', '%EO', 'LYMPH', '%LYMPH', '%MONO', 'MONO',
    'MPV', 'KREA', 'Na', 'K', '%IG', 'IG', 'eGFR-MDRD', 'eGFRCKD',
    'NEUT', '%NEUT', '%NRBC', 'NRBC', 'WPT', 'GLU', 'INR', 'PT',
    'CRP', 'WAPTT', 'APTT', 'ALT', 'BUN', 'AST', 'P-LCR'
]

Wszystkie examination_type values:
['Albumina', 'Albumina w moczu', 'Albumina w surowicy', 'Amylaza w moczu', 'Antystreptolizyna O (ASO)', 'Antystreptolizyny', 'Astrowirusy - antygen [kał]', 'Azot Mocznika', 'Azot mocznika w moczu', 'Azot mocznika we krwi', 'Beta-2-mikroglobulina', 'Beta-2-mikroglobulina we krwi', 'Białko C-reaktywne', 'Białko całkowite we krwi', 'Bilirubina bezpośrednia', 'Bilirubina całkowita we krwi', 'Biochemia', 'Chlorki (Cl)', 'Chlorki we krwi', 'Czas częściowej tromboplastyny po aktywacji', 'Czas protrombinowy', 'Czynnik reumatoidalny (RF)', 'Czynnik reumatoidalny w klasie IgM ( metoda turbidymetryczna)', 'Elektrolity', 'Elektrolity (potas, sód)', 'Enzymy', 'Erytroblasty', 'Fosfor nieorganiczny we krwi', 'Fosforany nieorganiczne', 'Glukoza', 'HPC Morfologia (produkt cytaferezy)', 'Hormony 2', 'INR', 'KTV- mocz', 'KTV- surowica', 'Koagulologia', 'Koagulologia- przeciwciała', 'Kreatynina', 'Kreatynina w moczu', 'Kreatynina we krwi', 'Kwas moczowy', 'Kwas moczowy w

In [ ]:
COLUMN_TO_EXAMINATIONS = {
    "MCH": ["Morfologia", "Morfologia z retikulocytami"],
    "MCHC": ["Morfologia", "Morfologia z retikulocytami"],
    "MCV": ["Morfologia", "Morfologia z retikulocytami"],
    "HGB": ["Morfologia", "Morfologia z retikulocytami"],
    "RBC": ["Morfologia", "Morfologia z retikulocytami"],
    "WBC": ["Morfologia", "Morfologia z retikulocytami"],
    "HCT": ["Morfologia", "Morfologia z retikulocytami"],
    "RDW": ["Morfologia", "Morfologia z retikulocytami"],
    "PLT": ["Morfologia", "Morfologia z retikulocytami"],
    "MPV": ["Morfologia", "Morfologia z retikulocytami"],

    "NEUT": ["Morfologia", "Rozmaz mikroskopowy krwi obwodowej"],
    "LYMPH": ["Morfologia", "Rozmaz mikroskopowy krwi obwodowej"],
    "MONO": ["Morfologia", "Rozmaz mikroskopowy krwi obwodowej"],
    "EO": ["Morfologia", "Rozmaz mikroskopowy krwi obwodowej"],
    "BAZO": ["Morfologia", "Rozmaz mikroskopowy krwi obwodowej"],

    "IG": ["Morfologia"],
    "NRBC": ["Morfologia"],

    "%NEUT": ["Morfologia"],
    "%LYMPH": ["Morfologia"],
    "%MONO": ["Morfologia"],
    "%EO": ["Morfologia"],
    "%BAZO": ["Morfologia"],
    "%IG": ["Morfologia"],
    "%NRBC": ["Morfologia"],

    "GLU": ["Glukoza"],

    "KREA": ["Kreatynina"],
    "eGFR-MDRD": ["Wskaźnik filtracji kłębuszkowej (wg MDRD)"],
    "eGFRCKD": ["Wskaźnik filtracji kłębuszkowej (wg CKD-EPI)"],

    "BUN": [
        "Azot mocznika we krwi",
        "Mocznik we krwi (parametr wyliczany)"
    ],

    "Na": ["Sód", "Elektrolity", "Elektrolity (potas, sód)"],
    "K": ["Potas", "Elektrolity", "Elektrolity (potas, sód)"],

    "CRP": ["Białko C-reaktywne"],

    "ALT": ["Transaminaza alaninowa"],
    "AST": ["Transaminaza asparaginianowa"],

    "INR": ["INR"],
    "PT": ["Czas protrombinowy", "Wskaźnik protrombinowy"],
    "APTT": [
        "Czas częściowej tromboplastyny po aktywacji",
        "Współczynnik APTT"
    ]
}

In [46]:
## create new csv without columns unit and norm for further analysis
def unpack_values(x):
    if pd.isna(x):
        return x

    # jeśli to string → próbuj sparsować
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except:
            return x

    # jeśli dict → rozpakuj values
    if isinstance(x, dict) and "values" in x:
        return ",".join(map(str, x["values"]))

    return x

def auto_unpack_values_dicts(df):
    df = df.copy()

    for col in df.columns:
        df[col] = df[col].apply(unpack_values)

    return df

def drop_unit_and_norm(df):
    cols_to_drop = [c for c in df.columns if c.endswith('-unit') or c.endswith('-norm')]
    return df.drop(columns=cols_to_drop)


def clean_and_save(df, path):
    df = df.copy()

    # 1. drop unit/norm
    df = drop_unit_and_norm(df)

    # 2. unpack ALL {'values': ...} dicts in any column
    df = auto_unpack_values_dicts(df)

    # 3. save
    df.to_csv(path, index=False)

    return df

kor_cleaned = clean_and_save(kor_dataset, "kor_cleaned1.csv")
neuro_cleaned = clean_and_save(neuro_dataset, "neuro_cleaned1.csv")

In [44]:
print("KOR patients:", kor_cleaned.columns.tolist())
print("\nNEURO patients:", neuro_cleaned.columns.tolist())

KOR patients: ['custom_id', 'patient_id', 'examination_date', 'patient_age', 'patient_sex', 'diagnosis_id', 'descriptive_result', 'examination_type', 'MCH', 'HGB', 'MCV', 'RBC', 'MCHC', 'PLT', 'HCT', 'WBC', 'RDW', '%BAZO', 'BAZO', 'EO', '%EO', 'LYMPH', '%LYMPH', '%MONO', 'MONO', 'MPV', 'KREA', 'Na', 'K', '%IG', 'IG', 'eGFR-MDRD', 'eGFRCKD', 'NEUT', '%NEUT', '%NRBC', 'NRBC', 'WPT', 'GLU', 'INR', 'PT', 'CRP', 'WAPTT', 'APTT', 'ALT', 'BUN', 'AST', 'P-LCR', 'nan_count']

NEURO patients: ['custom_id', 'patient_id', 'examination_date', 'patient_age', 'patient_sex', 'diagnosis_id', 'descriptive_result', 'examination_type', 'MCH', 'HGB', 'MCV', 'RBC', 'MCHC', 'PLT', 'HCT', 'WBC', 'RDW', '%BAZO', 'BAZO', 'EO', '%EO', 'LYMPH', '%LYMPH', '%MONO', 'MONO', 'MPV', 'KREA', 'Na', 'K', '%IG', 'IG', 'eGFR-MDRD', 'eGFRCKD', 'NEUT', '%NEUT', '%NRBC', 'NRBC', 'WPT', 'GLU', 'INR', 'PT', 'CRP', 'WAPTT', 'APTT', 'ALT', 'BUN', 'AST', 'P-LCR']
